# SQL Agent - NL2SQL 问答系统

基于 Chinook 数据库的自然语言到 SQL 转换代理

In [34]:

from typing import List, Dict, Any, Tuple, Union
from openai import OpenAI 
import re
import traceback
import pandas as pd
from sqlalchemy import create_engine, inspect, func, select, Table, MetaData

# 初始化 OpenAI 客户端（使用智谱 AI）
client = OpenAI(
    api_key="",
    base_url="https://open.bigmodel.cn/api/paas/v4/"
)

## 数据库解析器

In [35]:
class DBParser:
    '''数据库解析器 - 用于获取数据库结构和元数据'''
    def __init__(self, db_url: str) -> None:
        # 判断数据库类型
        if 'sqlite' in db_url:
            self.db_type = 'sqlite'
        elif 'mysql' in db_url:
            self.db_type = 'mysql'

        # 链接数据库
        self.engine = create_engine(db_url, echo=False)
        self.conn = self.engine.connect()
        self.db_url = db_url

        # 查看表名
        self.inspector = inspect(self.engine)
        self.table_names = self.inspector.get_table_names()

        self._table_fields = {}  # 数据表字段
        self.foreign_keys = []  # 数据库外键
        self._table_sample = {}  # 数据表样例

        # 依次对每张表的字段进行统计
        for table_name in self.table_names:
            print("Table ->", table_name)
            self._table_fields[table_name] = {}

            # 累计外键
            self.foreign_keys += [
                {
                    'constrained_table': table_name,
                    'constrained_columns': x['constrained_columns'],
                    'referred_table': x['referred_table'],
                    'referred_columns': x['referred_columns'],
                } for x in self.inspector.get_foreign_keys(table_name)
            ]

            # 获取当前表的字段信息
            table_instance = Table(table_name, MetaData(), autoload_with=self.engine)
            table_columns = self.inspector.get_columns(table_name)
            self._table_fields[table_name] = {x['name']: x for x in table_columns}

            # 对当前字段进行统计
            for column_meta in table_columns:
                column_instance = getattr(table_instance.columns, column_meta['name'])

                # 统计unique
                query = select(func.count(func.distinct(column_instance)))
                distinct_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['distinct'] = distinct_count

                # 统计most frequency value
                field_type = str(self._table_fields[table_name][column_meta['name']]['type'])
                if 'text' in field_type.lower() or 'char' in field_type.lower():
                    query = (
                        select(column_instance, func.count().label('count'))
                        .group_by(column_instance)
                        .order_by(func.count().desc())
                        .limit(1)
                    )
                    top1_value = self.conn.execute(query).fetchone()[0]
                    self._table_fields[table_name][column_meta['name']]['mode'] = top1_value

                # 统计missing个数
                query = select(func.count()).filter(column_instance == None)
                nan_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['nan_count'] = nan_count

                # 统计max
                query = select(func.max(column_instance))
                max_value = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['max'] = max_value

                # 统计min
                query = select(func.min(column_instance))
                min_value = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['min'] = min_value

                # 任意取值
                query = select(column_instance).limit(10)
                random_value = self.conn.execute(query).all()
                random_value = [x[0] for x in random_value]
                random_value = [str(x) for x in random_value if x is not None]
                random_value = list(set(random_value))
                self._table_fields[table_name][column_meta['name']]['random'] = random_value[:3]

            # 获取表样例（第一行）
            query = select(table_instance)
            self._table_sample[table_name] = pd.DataFrame([self.conn.execute(query).fetchone()])
            self._table_sample[table_name].columns = [x['name'] for x in table_columns]

    def get_table_fields(self, table_name) -> pd.DataFrame:
        '''获取表字段信息'''
        return pd.DataFrame.from_dict(self._table_fields[table_name]).T

    def get_data_relations(self) -> pd.DataFrame:
        '''获取数据库链接信息（主键和外键）'''
        return pd.DataFrame(self.foreign_keys)

    def get_table_sample(self, table_name) -> pd.DataFrame:
        '''获取数据表样例'''
        return self._table_sample[table_name]

    def check_sql(self, sql) -> Tuple[bool, str]:
        '''检查sql是否合理'''
        try:
            # 处理可能的多条SQL语句（取第一条执行）
            sql_statements = [s.strip() for s in sql.split(';') if s.strip()]
            if not sql_statements:
                return False, 'Empty SQL'
            
            # 只执行第一条SQL进行验证
            first_sql = sql_statements[0]
            if not first_sql.upper().startswith(('SELECT', 'INSERT', 'UPDATE', 'DELETE')):
                first_sql += ';'
            
            result = self.conn.execute(text(first_sql))
            return True, 'ok'
        except Exception as e:
            err_msg = traceback.format_exc()
            return False, err_msg

    def execute_sql(self, sql) -> List:
        '''运行SQL - 支持单条或多条SQL语句'''
        from sqlalchemy import text
        
        # 处理可能的多条SQL语句
        sql_statements = [s.strip() for s in sql.split(';') if s.strip()]
        
        if not sql_statements:
            return []
        
        # 如果只有一条SQL，直接执行
        if len(sql_statements) == 1:
            result = self.conn.execute(text(sql_statements[0]))
            return result.fetchall()
        
        # 如果有多条SQL，依次执行并合并结果
        all_results = []
        for stmt in sql_statements:
            try:
                result = self.conn.execute(text(stmt))
                all_results.append(result.fetchall())
            except Exception as e:
                all_results.append(f'Error executing: {stmt} - {str(e)}')
        
        return all_results

## SQL Agent 实现

In [36]:
class SQLAgent:
    '''SQL Agent - 自然语言转SQL查询代理'''
    
    def __init__(self, db_parser: DBParser):
        self.db_parser = db_parser
        self.history = []  # 对话历史
        self.retry_time = 3  # 重试次数
        
        # 提示词模板
        self.system_prompt = """你是一个专业的SQL专家，擅长将自然语言问题转换为SQL查询语句。

数据库包含以下表：{table_list}

请根据用户的问题，生成对应的SQL查询语句。

要求：
1. **只输出单条SQL语句**，不要有多条SQL
2. 如果需要查询多个数据，使用 UNION、JOIN 或子查询合并为一条SQL
3. SQL语句应该可以直接在SQLite数据库中执行
4. 如果问题无法用SQL回答，请返回 "无法回答"
5. 使用标准的SQL语法
6. 不要在SQL末尾添加分号后的其他内容"""

        self.schema_prompt = """以下是数据库的详细结构信息：

{schema_info}

表之间的关系（外键）：
{relations_info}

请基于以上信息生成SQL查询。"""

        self.reflection_prompt = """之前生成的SQL执行失败，错误信息为：
{error_message}

原始SQL：
{original_sql}

请修正SQL语句，只输出修正后的SQL，不要有其他内容。"""

        self.answer_prompt = """你是一个数据分析师，请将SQL查询结果用自然语言回答用户的问题。

用户问题：{question}
SQL查询：{sql}
查询结果：{result}

请用简洁、清晰的自然语言回答。如果有多条查询结果，请分别说明。"""

    def action(self, question: str) -> str:
        '''
        处理用户提问
        
        Args:
            question: 用户的自然语言问题
            
        Returns:
            自然语言答案
        '''
        print(f"\n{'='*60}")
        print(f"用户问题: {question}")
        print(f"{'='*60}\n")
        
        # 步骤1: 生成数据库schema信息
        schema_info = self._get_schema_info()
        relations_info = self._get_relations_info()
        
        # 步骤2: 构建完整的系统提示
        system_content = self.system_prompt.format(
            table_list=", ".join(self.db_parser.table_names)
        ) + "\n\n" + self.schema_prompt.format(
            schema_info=schema_info,
            relations_info=relations_info
        )
        
        # 步骤3: 生成SQL（带重试机制）
        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": question}
        ]
        
        sql = None
        for retry_idx in range(self.retry_time):
            try:
                # 调用LLM生成SQL
                response = self._call_llm(messages)
                sql = self._extract_sql(response)
                
                if not sql or sql == "无法回答":
                    return "抱歉，我无法理解您的问题或该问题无法通过数据库查询回答。"
                
                print(f"生成的SQL (尝试 {retry_idx + 1}):\n{sql}\n")
                
                # 验证并执行SQL
                success, error_msg = self.db_parser.check_sql(sql)
                
                if success:
                    result = self.db_parser.execute_sql(sql)
                    print(f"查询结果: {result}\n")
                    
                    # 将结果转换为自然语言
                    answer = self._generate_natural_answer(question, sql, result)
                    
                    # 记录历史
                    self.history.append({
                        "question": question,
                        "sql": sql,
                        "result": result,
                        "answer": answer
                    })
                    
                    return answer
                else:
                    print(f"SQL执行失败: {error_msg}\n")
                    # 添加反思提示
                    messages.append({
                        "role": "user",
                        "content": self.reflection_prompt.format(
                            error_message=error_msg,
                            original_sql=sql
                        )
                    })
            except Exception as e:
                print(f"发生错误: {str(e)}\n")
                messages.append({
                    "role": "user",
                    "content": f"发生错误: {str(e)}，请重新生成SQL。"
                })
        
        return "抱歉，经过多次尝试仍无法生成正确的SQL查询。"

    def _get_schema_info(self) -> str:
        '''获取数据库schema信息'''
        schema_parts = []
        for table_name in self.db_parser.table_names:
            fields_df = self.db_parser.get_table_fields(table_name)
            sample_df = self.db_parser.get_table_sample(table_name)
            
            schema_part = f"\n表名: {table_name}\n"
            schema_part += f"字段信息:\n{fields_df[['name', 'type', 'nullable']].to_string()}\n"
            schema_part += f"样例数据:\n{sample_df.to_string()}\n"
            schema_parts.append(schema_part)
        
        return "\n".join(schema_parts)

    def _get_relations_info(self) -> str:
        '''获取表关系信息'''
        relations_df = self.db_parser.get_data_relations()
        if relations_df.empty:
            return "无外键关系"
        return relations_df.to_string()

    def _call_llm(self, messages: List[Dict]) -> str:
        '''调用大语言模型'''
        try:
            completion = client.chat.completions.create(
                model="glm-4-flash",
                messages=messages,
                temperature=0.1,
                top_p=0.7
            )
            return completion.choices[0].message.content
        except Exception as e:
            print(f"调用LLM失败: {str(e)}")
            raise

    def _extract_sql(self, text: str) -> str:
        '''从LLM响应中提取SQL语句'''
        if not text:
            return ""
        
        # 尝试从markdown代码块中提取
        pattern = r'```sql\s*(.*?)\s*```|```\s*(.*?)\s*```'
        matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
        
        if matches:
            # 取第一个匹配的非空组
            for match in matches:
                sql = match[0] if match[0] else match[1]
                if sql:
                    return sql.strip()
        
        # 如果没有找到代码块，直接返回清理后的文本
        sql = text.strip()
        # 移除可能的前缀说明
        sql = re.sub(r'^.*?(SELECT|INSERT|UPDATE|DELETE|CREATE|DROP)', r'\1', sql, flags=re.IGNORECASE | re.DOTALL)
        
        return sql

    def _generate_natural_answer(self, question: str, sql: str, result: List) -> str:
        '''将SQL结果转换为自然语言答案'''
        try:
            messages = [
                {"role": "user", "content": self.answer_prompt.format(
                    question=question,
                    sql=sql,
                    result=str(result)
                )}
            ]
            answer = self._call_llm(messages)
            return answer
        except:
            # 如果生成自然语言失败，直接返回原始结果
            return f"查询结果: {result}"

## 初始化并使用 Agent

In [37]:
# 初始化数据库解析器
print("正在初始化数据库解析器...")
parser = DBParser('sqlite:///./chinook.db')
print(f"\n数据库包含 {len(parser.table_names)} 张表")
print(f"表名列表: {parser.table_names}\n")

正在初始化数据库解析器...
Table -> albums
Table -> artists
Table -> customers
Table -> employees
Table -> genres
Table -> invoice_items
Table -> invoices
Table -> media_types
Table -> playlist_track
Table -> playlists
Table -> tracks

数据库包含 11 张表
表名列表: ['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'invoices', 'media_types', 'playlist_track', 'playlists', 'tracks']



In [38]:
# 创建 SQL Agent
agent = SQLAgent(parser)

### 测试问题 1: 数据库中总共有多少张表

In [39]:
question1 = "数据库中总共有多少张表？"
answer1 = agent.action(question1)
print(f"\n最终答案: {answer1}")


用户问题: 数据库中总共有多少张表？

生成的SQL (尝试 1):
SELECT COUNT(*) FROM sqlite_master WHERE type='table';

查询结果: [(13,)]


最终答案: 数据库中总共有13张表。


### 测试问题 2: 员工表中有多少条记录

In [40]:
question2 = "员工表中有多少条记录？"
answer2 = agent.action(question2)
print(f"\n最终答案: {answer2}")


用户问题: 员工表中有多少条记录？

生成的SQL (尝试 1):
SELECT COUNT(*) FROM employees;

查询结果: [(8,)]


最终答案: 员工表中总共有8条记录。


### 测试问题 3: 在数据库中所有客户个数和员工个数分别是多少

In [41]:
question3 = "在数据库中所有客户个数和员工个数分别是多少？"
answer3 = agent.action(question3)
print(f"\n最终答案: {answer3}")


用户问题: 在数据库中所有客户个数和员工个数分别是多少？

生成的SQL (尝试 1):
SELECT 
    (SELECT COUNT(*) FROM customers) AS CustomersCount,
    (SELECT COUNT(*) FROM employees) AS EmployeesCount;

查询结果: [(59, 8)]


最终答案: 数据库中，客户总数为59个，员工总数为8个。


## 更多测试问题

In [42]:
# 额外测试：查询某个艺术家的专辑数量
question4 = "AC/DC乐队有多少张专辑？"
answer4 = agent.action(question4)
print(f"\n最终答案: {answer4}")


用户问题: AC/DC乐队有多少张专辑？

生成的SQL (尝试 1):
SELECT COUNT(*) AS NumberOfAlbums
FROM albums
JOIN artists ON albums.ArtistId = artists.ArtistId
WHERE artists.Name = 'AC/DC';

查询结果: [(2,)]


最终答案: AC/DC乐队共有2张专辑。


In [43]:
# 额外测试：查询最贵的歌曲
question5 = "最贵的歌曲是什么？价格是多少？"
answer5 = agent.action(question5)
print(f"\n最终答案: {answer5}")


用户问题: 最贵的歌曲是什么？价格是多少？

生成的SQL (尝试 1):
SELECT t.Name AS TrackName, t.UnitPrice AS Price
FROM tracks t
ORDER BY t.UnitPrice DESC
LIMIT 1;

查询结果: [('Battlestar Galactica: The Story So Far', 1.99)]


最终答案: 最贵的歌曲是《Battlestar Galactica: The Story So Far》，价格为1.99美元。


In [44]:
# 额外测试：查询客户来自哪些国家
question6 = "客户都来自哪些不同的国家？"
answer6 = agent.action(question6)
print(f"\n最终答案: {answer6}")


用户问题: 客户都来自哪些不同的国家？

生成的SQL (尝试 1):
SELECT DISTINCT Country
FROM customers;

查询结果: [('Brazil',), ('Germany',), ('Canada',), ('Norway',), ('Czech Republic',), ('Austria',), ('Belgium',), ('Denmark',), ('USA',), ('Portugal',), ('France',), ('Finland',), ('Hungary',), ('Ireland',), ('Italy',), ('Netherlands',), ('Poland',), ('Spain',), ('Sweden',), ('United Kingdom',), ('Australia',), ('Argentina',), ('Chile',), ('India',)]


最终答案: 客户来自以下不同的国家：巴西、德国、加拿大、挪威、捷克共和国、奥地利、比利时、丹麦、美国、葡萄牙、法国、芬兰、匈牙利、爱尔兰、意大利、荷兰、波兰、西班牙、瑞典、英国、澳大利亚、阿根廷、智利和印度。
